## Pierwszy duży projekt - Cyfrowy Bliźniak

### Ale najpierw: przedstawiam Pushover

Pushover to fajne narzędzie do wysyłania powiadomień push na Twój telefon.

Jest bardzo łatwe w konfiguracji i instalacji!

Po prostu wejdź na https://pushover.net/ i kliknij 'Login or Signup' w prawym górnym rogu, żeby założyć darmowe konto i utworzyć swoje klucze API.

Po rejestracji, na ekranie głównym kliknij "Create an Application/API Token", nadaj dowolną nazwę (np. Agents) i kliknij Create Application.

Potem dodaj 2 linie do swojego pliku `.env`:

PUSHOVER_USER=_wpisz klucz, który jest w prawym górnym rogu ekranu głównego Pushover i prawdopodobnie zaczyna się od u_  
PUSHOVER_TOKEN=_wpisz klucz, gdy klikniesz w swoją nową aplikację o nazwie Agents (lub jakiejkolwiek innej) i prawdopodobnie zaczyna się od a_

Pamiętaj, żeby zapisać plik `.env` i uruchomić `load_dotenv(override=True)` po zapisaniu, żeby ustawić zmienne środowiskowe.

Na koniec kliknij "Add Phone, Tablet or Desktop", żeby zainstalować na swoim telefonie.

## Uwaga - zmiana względem filmów

W filmie wdrażam bliźniaka za darmo na HuggingFace Spaces. HuggingFace ostatnio przestał to wspierać za darmo!

Jest darmowa alternatywa, i wyjaśniam ją oraz podaję instrukcje dalej w tym labie.

In [ ]:
# importy

from dotenv import load_dotenv
from anthropic import Anthropic
import json
import os
import requests
from pypdf import PdfReader
import gradio as gr

In [ ]:
# Zwyczajowy start

load_dotenv(override=True)
anthropic = Anthropic()  # klient automatycznie odczyta klucz z ANTHROPIC_API_KEY w .env

In [ ]:
# Dla pushover

pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    if pushover_user.startswith("u"):
        print("Pushover user found and looks good")
    else:
        print("Pushover user found but doesn't start with u")
else:
    print("Pushover user not found")

if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover token found and looks good")
    else:
        print("Pushover token found but doesn't start with a")
else:
    print("Pushover token not found")

In [ ]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [ ]:
push("HEY!!")

In [ ]:
def record_user_details(email, name="Name not provided", notes="not provided"):
    push(f"Recording interest from {name} with email {email} and notes {notes}")
    return "OK"

In [ ]:
def record_unknown_question(question):
    push(f"Recording {question} asked that I couldn't answer")
    return "OK"

In [ ]:
record_user_details_json = {
    "name": "record_user_details",
    "description": "Użyj tego narzędzia, żeby zapisać, że użytkownik jest zainteresowany kontaktem i podał adres email",
    "input_schema": {  # Anthropic używa klucza input_schema, nie parameters jak OpenAI
        "type": "object",
        "properties": {
            "email": {"type": "string", "description": "Adres email tego użytkownika"},
            "name": {"type": "string", "description": "Imię użytkownika, jeśli je podał"},
            "notes": {"type": "string", "description": "Dodatkowe informacje o rozmowie, warte zapisania jako kontekst"
            }
        },
        "required": ["email"],
        "additionalProperties": False
    }
}

In [ ]:
record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "Zawsze użyj tego narzędzia, żeby zapisać każde pytanie, na które nie potrafiłeś odpowiedzieć, bo nie znałeś odpowiedzi",
    "input_schema": {
        "type": "object",
        "properties": {
            "question": {"type": "string", "description": "Pytanie, na które nie udało się odpowiedzieć"},
        },
        "required": ["question"],
        "additionalProperties": False
    }
}

In [ ]:
tools = [record_user_details_json, record_unknown_question_json]  # Anthropic przyjmuje płaską listę definicji narzędzi, bez opakowania {"type": "function", "function": ...} jak w OpenAI

In [ ]:
tools

In [ ]:
# Ta funkcja może przyjąć listę bloków tool_use i je wykonać. To jest ta instrukcja IF!!

def handle_tool_calls_with_manual_if(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.name  # blok tool_use ma name bezpośrednio, nie zagnieżdżone w .function jak w OpenAI
        arguments = tool_call.input  # input jest już sparsowanym dict, nie JSON-stringiem jak tool_call.function.arguments w OpenAI
        print(f"Tool called: {tool_name}", flush=True)

        # WIELKA INSTRUKCJA IF!!!

        if tool_name == "record_user_details":
            result = record_user_details(**arguments)
        elif tool_name == "record_unknown_question":
            result = record_unknown_question(**arguments)

        results.append({
            "type": "tool_result",
            "tool_use_id": tool_call.id,  # musi się zgadzać z id bloku tool_use
            "content": json.dumps(result)
        })
    return results

## Używanie wbudowanej funkcji globals() w Pythonie

Python ma słownik, który daje nam dostęp do wszystkich globalnych funkcji.

Dygresja: przy wdrożeniu na pewno użyjemy tego w bardziej zabezpieczony sposób..

In [ ]:
globals()["record_unknown_question"]("this is a really hard question")

In [ ]:
# To daje nam bardziej elegancki sposób, który omija instrukcję IF.

def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.name
        arguments = tool_call.input
        print(f"Tool called: {tool_name}", flush=True)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else "No tool found"
        results.append({
            "type": "tool_result",
            "tool_use_id": tool_call.id,
            "content": json.dumps(result)
        })
    return results

In [ ]:
reader = PdfReader("twin-pm/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

with open("twin-pm/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [ ]:
system_prompt = f"""

# Twoja rola

Jesteś cyfrowym bliźniakiem działającym na stronie internetowej, rozmawiającym z jej odwiedzającymi.
Reprezentujesz osobę, do której należy ta strona.
Odpowiadasz na pytania dotyczące jej kariery, doświadczenia, umiejętności i historii zawodowej.

Oto szczegóły dotyczące osoby, którą reprezentujesz:

{summary}

Jeśli zostaniesz o to zapytany, wyjaśnij jasno, że jesteś AI będącym cyfrowym bliźniakiem tej osoby.

# Kontekst

Oto podsumowanie profilu LinkedIn tej osoby, dzięki któremu możesz odpowiadać na pytania:

{linkedin}

# Zasady

Angażuj się w rozmowę z użytkownikiem. Bądź profesjonalny i przystępny, jakbyś rozmawiał z potencjalnym klientem albo przyszłym pracodawcą, który trafił na tę stronę.
Odpowiadaj wyłącznie na pytania dotyczące kariery, doświadczenia, umiejętności i historii zawodowej.
Jeśli użytkownik zapyta o coś niezwiązanego, sprowadź rozmowę z powrotem na tematy zawodowe.

Zawsze pozostawaj w roli cyfrowego bliźniaka osoby, którą reprezentujesz. Reprezentuj tę osobę.

Jeśli użytkownik chciałby się skontaktować, poproś o jego adres email i użyj swojego narzędzia, żeby zapisać ten email do dalszego kontaktu.

WAŻNE:
Jeśli nie znasz odpowiedzi, użyj swojego narzędzia, żeby zapisać pytanie, a potem powiedz użytkownikowi, że nie wiesz. Nigdy nie zmyślaj odpowiedzi.
"""


In [ ]:
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]  # Gradio może dawać dodatkowe klucze w historii (np. metadata) - Anthropic akceptuje tylko role/content
    messages = history + [{"role": "user", "content": message}]
    response = anthropic.messages.create(model="claude-haiku-4-5", max_tokens=16000, system=system_prompt, tools=tools, messages=messages)  # system_prompt jako top-level parametr, nie wpis w messages

    while response.stop_reason == "tool_use":  # pętla trwa, dopóki Claude chce użyć narzędzia
        tool_calls = [block for block in response.content if block.type == "tool_use"]
        results = handle_tool_calls(tool_calls)
        messages.append({"role": "assistant", "content": response.content})  # cała odpowiedź assistant (razem z blokiem tool_use) wraca do historii
        messages.append({"role": "user", "content": results})  # wszystkie wyniki narzędzi w JEDNEJ wiadomości user - Claude oczekuje ich razem, nie po jednej na wiadomość
        response = anthropic.messages.create(model="claude-haiku-4-5", max_tokens=16000, system=system_prompt, tools=tools, messages=messages)

    return next(block.text for block in response.content if block.type == "text")

In [ ]:
gr.ChatInterface(chat).launch(inbrowser=True)

## Zamiana na moduły Pythona

Zamieniłem kod z labu na moduły Pythona; to świetna praktyka po zakończeniu eksperymentów w Notatniku.

Cały powyższy kod mógłbyś umieścić w 1 skrypcie Pythona. Ale ładniej jest zorganizować kod w różnych modułach dla różnych zagadnień, i tak właśnie zrobiłem:

`context.py` ładuje statyczne dane i konstruuje System Prompt

`tools.py` zawiera cały kod do zarządzania i wywoływania narzędzi, wraz z ich powiązanym json

`app.py` zawiera aplikację Gradio i wywołanie OpenAI.

`styles.py` zawiera style do zastosowania w Gradio i zostały napisane w całości przez Claude Code!

Możesz spróbować zrobić to sam, a potem porównać ze swoimi wersjami.

Żeby to wypróbować, otwórz terminal w Cursorze:

`cd 1_foundations`  
`cd twin`  
`uv run app.py`

# WSTRZYMAĆ PRASĘ! Uwaga...

Od 9 lipca 2026, HuggingFace nagle przestał pozwalać na darmowe wdrażanie aplikacji Gradio na HuggingFace Spaces.

To dość nieprzyjemna niespodzianka!

Spodziewam się, że mogą wycofać się z tej decyzji. Tymczasem oto darmowa alternatywa: użycie Render.

Pełne instrukcje znajdziesz w [pliku RENDER_INSTRUCTIONS w tym katalogu](RENDER_INSTRUCTIONS.md)

Jeśli nie masz nic przeciwko płaceniu za HuggingFace, oryginalne instrukcje są poniżej.

A także oto instrukcje mojego cyfrowego bliźniaka, który działa bardzo tanio na fly.io:  
https://edwarddonner.com/avatar

Dzięki mojemu bliźniakowi nie tylko możesz powiadomić mnie Pushem, ale też porozmawiać z prawdziwym mną! Oto film o tym, jak go zrobiłem, wraz z instrukcjami, gdybyś też chciał go zrobić. Zacząłem od tej aplikacji Career Conversations.  
https://youtu.be/srlhW4H-Gtg

## Oryginalne instrukcje z HF Spaces (już nie za darmo)

Wdrożymy na HuggingFace Spaces.

Zanim zaczniesz: pamiętaj, żeby zaktualizować pliki w katalogu `twin` - swój profil LinkedIn i summary.txt - żeby mówiły o Tobie!

Sprawdź też, czy w katalogu twin nie ma pliku README. Jeśli jest, usuń go. Proces wdrożenia sam tworzy tam nowy plik README.

## Wdrożenie Część 1: HuggingFace

1. Wejdź na https://huggingface.co i załóż konto  
2. Z menu Avatar w prawym górnym rogu wybierz Access Tokens. Wybierz "Create New Token". Nadaj mu uprawnienia WRITE - musi mieć uprawnienia WRITE! Zachowaj swój nowy klucz.  
3. W Terminalu Cursora uruchom: `uvx hf auth login --token YOUR_TOKEN_HERE`, np. `uvx hf auth login --token hf_xxxxxx`, żeby zalogować się z poziomu wiersza poleceń swoim kluczem. Potem uruchom `uvx hf auth whoami`, żeby sprawdzić, czy jesteś zalogowany  
4. Weź swój nowy token i dodaj go do pliku .env: `HF_TOKEN=hf_xxx` na przyszłość

## Wdrożenie Część 2: Push!

1. Wejdź do katalogu twin: `cd 1_foundations`, a potem `cd twin`
2. Z katalogu twin wpisz: `uv run gradio deploy` 
3. Postępuj zgodnie z instrukcjami, wybierając wartości domyślne: nazwij to `twin`, wskaż app.py, wybierz cpu-basic jako sprzęt, odpowiedz "No" na pytanie o potrzebę podania sekretów i "no" na github actions.  

### Wdrożenie Część 3: Sekrety

1. Wejdź na https://huggingface.co, kliknij swój Avatar, przejdź do swojego profilu, wybierz Space
2. Wejdź w menu 3 kropek i wybierz Settings
3. Przewiń w dół do sekcji Variables and Secrets
4. Naciśnij "New Secret" (nie New Variable) i wpisz nazwę `OPENAI_API_KEY` oraz wartość swojego klucza z pliku .env (albo użyj odpowiedniego klucza dla swojego LLM). Uważaj, żeby zrobić to poprawnie!
5. Powtórz dla `PUSHOVER_USER` i `PUSHOVER_TOKEN` z pliku .env
6. Bliżej góry ustawień kliknij "Restart space", żeby go zrestartować
7. Kliknij App bliżej góry, żeby wrócić do aplikacji, i po jej restarcie - ciesz się!

### Osadzanie na innej stronie

Żeby osadzić to na innej stronie, wybierz "Embed this space" z menu trzech kropek.

### Rozwiązywanie problemów

Jeśli dostaniesz błąd gradio, spróbuj otworzyć logi (przycisk obok menu 3 kropek).  
Spróbuj dodać więcej informacji debugowych, zwłaszcza wokół swoich kluczy.

### Ponowne wdrażanie space'a

Po prostu uruchom `uv run gradio deploy` z katalogu twin. Może być konieczne usunięcie pliku README.md, który stworzyło tam Gradio, jeśli chcesz ponownie nazwać swój space.

### Usuwanie space'a

Z menu 3 kropek wybierz ekran Settings, a na dole znajdziesz opcję Delete.

Więcej informacji o wdrożeniu:

https://www.gradio.app/guides/sharing-your-app#hosting-on-hf-spaces

### Mój Cyfrowy Bliźniak

Poświęciłem trochę czasu, żeby przenieść mojego Cyfrowego bliźniaka na wyższy poziom!  
Oto on:   
https://edwarddonner.com/avatar

Dzięki mojemu bliźniakowi nie tylko możesz powiadomić mnie Pushem, ale też porozmawiać z prawdziwym mną! Oto film o tym, jak go zrobiłem, wraz z instrukcjami, gdybyś też chciał go zrobić. Zacząłem od tej aplikacji Career Conversations.  
https://youtu.be/srlhW4H-Gtg


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Ćwiczenie</h2>
            <span style="color:#ff7800;">• Przede wszystkim, wdróż to dla siebie! To realne, wartościowe narzędzie - CV przyszłości..<br/>
            • Następnie popraw materiały - dodaj lepszy kontekst o sobie. Jeśli znasz RAG, dodaj bazę wiedzy o sobie.<br/>
            • Dodaj więcej narzędzi! Mógłbyś mieć bazę SQL z częstymi pytaniami i odpowiedziami, z której LLM mógłby czytać i do której mógłby pisać?<br/>
            • Wprowadź Evaluatora z ćwiczenia z Dnia 4 i dodaj inne wzorce Agentic.<br/>
            • Niektórzy kursanci dodali integrację z Telegramem, żebyś mógł na żywo rozmawiać z ludźmi na swojej stronie, razem ze swoim bliźniakiem!
            </span>
        </td>
    </tr>
</table>

### Przykładowe rozwiązanie (niezależne od ćwiczenia powyżej)

Integracja z Telegramem przez long polling Telegram Bot API - reużywa istniejącą funkcję `chat()` z tego notatnika, z osobną historią rozmowy dla każdego czatu na Telegramie. Wymaga `TELEGRAM_BOT_TOKEN` w pliku `.env`; token uzyskasz, pisząc `/newbot` do [@BotFather](https://t.me/BotFather) na Telegramie. Uruchom ostatnią komórkę (`run_telegram_bot()`), żeby zacząć nasłuchiwać - działa, dopóki nie przerwiesz komórki (Kernel -> Interrupt).

In [ ]:
import time  # potrzebne do przerwy między odpytywaniem Telegrama (time.sleep) w pętli long-pollingu niżej

telegram_bot_token = os.getenv("TELEGRAM_BOT_TOKEN")  # token bota utworzony przez @BotFather na Telegramie (komenda /newbot), dodaj go do .env
telegram_api_url = f"https://api.telegram.org/bot{telegram_bot_token}"  # bazowy URL Telegram Bot API - każda metoda (getUpdates, sendMessage) doklejana jest na końcu

if telegram_bot_token:
    print("Telegram bot token found")
else:
    print("Telegram bot token not found")

In [ ]:
def send_telegram_message(chat_id, text):
    payload = {"chat_id": chat_id, "text": text}  # chat_id identyfikuje konkretną rozmowę na Telegramie, do której wysyłamy odpowiedź
    requests.post(f"{telegram_api_url}/sendMessage", data=payload)  # metoda sendMessage Telegram Bot API

In [ ]:
telegram_histories = {}  # osobna historia rozmowy per chat_id, żeby wielu użytkowników Telegrama mogło rozmawiać z bliźniakiem niezależnie od siebie

def handle_telegram_update(update):
    message = update.get("message")  # interesują nas tylko aktualizacje zawierające zwykłą wiadomość, nie np. edycje czy reakcje
    if not message or "text" not in message:  # pomiń np. zdjęcia, naklejki albo zdarzenia dołączenia do grupy - nie mają czego przekazać do chat()
        return
    chat_id = message["chat"]["id"]  # unikalny identyfikator konkretnej rozmowy na Telegramie
    text = message["text"]  # treść wiadomości od użytkownika
    history = telegram_histories.setdefault(chat_id, [])  # przy pierwszej wiadomości z danego chat_id zakłada nową, pustą historię
    reply = chat(text, history)  # ta sama funkcja chat() co w Gradio wyżej w notatniku - identyczna logika, inny kanał dostarczenia
    history.append({"role": "user", "content": text})  # dopisz wiadomość użytkownika do historii tego czatu
    history.append({"role": "assistant", "content": reply})  # dopisz odpowiedź bliźniaka do historii tego czatu
    send_telegram_message(chat_id, reply)  # odeślij odpowiedź z powrotem na Telegrama

In [ ]:
def run_telegram_bot(poll_interval=1):
    print("Telegram bot uruchomiony - napisz do swojego bota na Telegramie. Zatrzymaj przerywając tę komórkę (Kernel -> Interrupt).")
    offset = None  # id ostatniej przetworzonej aktualizacji + 1 - bez tego Telegram wciąż zwracałby te same, już obsłużone wiadomości
    try:
        while True:
            params = {"timeout": 30}  # long polling: Telegram trzyma połączenie otwarte do 30s i odpowiada od razu, gdy przyjdzie nowa wiadomość, zamiast pytać w kółko na pusto
            if offset is not None:
                params["offset"] = offset  # poproś tylko o aktualizacje nowsze niż ostatnio przetworzona
            response = requests.get(f"{telegram_api_url}/getUpdates", params=params, timeout=35)  # timeout requests trochę większy niż timeout long-pollingu Telegrama, żeby nie ucinać połączenia przedwcześnie
            data = response.json()
            if not data.get("ok"):  # błąd Telegrama (np. zły token) przychodzi jako HTTP 200 z {"ok": false, ...}, nie jako wyjątek
                print(f"Telegram API zwrócił błąd: {data}")
                break
            for update in data["result"]:  # lista nowych aktualizacji od ostatniego zapytania
                handle_telegram_update(update)
                offset = update["update_id"] + 1  # przesuń offset za każdą przetworzoną aktualizację
            time.sleep(poll_interval)  # krótka przerwa między kolejnymi zapytaniami getUpdates
    except KeyboardInterrupt:  # pozwala łagodnie zatrzymać bota przerwaniem kernela zamiast zabijania procesu
        print("Telegram bot zatrzymany")

In [ ]:
run_telegram_bot()

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Implikacje komercyjne</h2>
            <span style="color:#00bfff;">Poza oczywistym zastosowaniem (Twoje CV przyszłości), ma to zastosowania biznesowe w każdej sytuacji, gdzie potrzebujesz asystenta AI z wiedzą dziedzinową i zdolnością interakcji z prawdziwym światem.
            </span>
        </td>
    </tr>
</table>